# Flow-Aware Temporal Pattern Mining for Multi-Stage Network Intrusion Detection

**13-stage pipeline on CIC-IDS2017** -- session reconstruction, behavioral event
encoding, sequential pattern mining (FP-Growth + PrefixSpan), a temporal
attack-state graph, multi-model evidence fusion, and an explainable risk
decision layer.

This notebook drives the `nids` package in `../src/nids`. It implements the
methodology **as corrected by the senior reviewer pass**, not the original
draft. The corrections baked into the code (see each stage's markdown cell
for the specific citation):

1. **Chronological, session-aware split** (Days 1-2 train / Day 3 val / Days
   4-5 test) -- never a random stratified split.
2. **SMOTE-KNN restricted to RF & XGBoost**, and only for classes with
   **>= 50** training samples. The LSTM, FP-Growth, PrefixSpan and the
   attack-state graph train on **original data only, always**.
3. **Class weighting is the primary imbalance strategy**, applied to every
   model including the LSTM's loss.
4. **Fusion weights are calibrated once on validation and frozen** -- never
   recomputed from live/rolling F1 during inference (that would need
   test-time labels).
5. **Risk_t's coefficients are learned** by a logistic regression
   meta-learner fit on validation, not hand-picked.
6. **Heartbleed / Infiltration are never merged** into a rare-class bucket.

> **Before you run this on the real dataset**, read the Stage 1 cell below
> about the zero-training-count caveat for Heartbleed/Infiltration under
> this split -- it changes how you should report those two classes.

## Setup

1. `pip install -r ../requirements.txt`
2. Point `DATA_DIR` below at your local CIC-IDS2017 folder
   (default: `D:\IDSPROJECT2026\CIC-IDS2017`).


In [ ]:
import sys, logging
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nids import (
    config, data_loading, preprocessing, features, sessions, events,
    models, fusion, pattern_mining, attack_graph, risk, streaming,
    explainability, metrics, pipeline, ablation,
)

# --- Point this at your dataset ------------------------------------------
DATA_DIR = config.DATA_DIR  # default: D:\IDSPROJECT2026\CIC-IDS2017
# DATA_DIR = Path(r"D:\IDSPROJECT2026\CIC-IDS2017")  # <- uncomment / edit if different

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)


## Stage 1-2: Data Acquisition, Cleaning & Session-Aware Chronological Split

CIC-IDS2017's 7-8 daily CSVs are auto-discovered by weekday name, tagged with
provenance (`day`, `day_order`, `source_file` -- **metadata only, never
features**, Review §2), chronologically sorted, and cleaned (Inf -> NaN ->
drop, CIC-IDS2017's known ~1.8% corrupt rows).

**Split (Review §5, corrected from a naive random split):**
Train = Monday+Tuesday, Validation = Wednesday, Test = Thursday+Friday.
Because a session can never span a day boundary (Stage 4), this day-level
split guarantees no session straddles train/val/test -- the leakage bug a
random split would introduce.

**Read this before you interpret your results:** in the real CIC-IDS2017
release, Heartbleed flows exist **only** in the Wednesday file and
Infiltration flows **only** in the Thursday file. Under this split that
means Heartbleed has **zero** training examples (it is validation-only) and
Infiltration has **zero** training examples (it is test-only) -- a model
cannot learn a decision boundary for a class it never saw in training. The
cell below runs `pipeline.check_zero_shot_classes` and will warn you loudly
if this applies to your data. Report it exactly as Review §4/§26-F
recommends: honestly, as a limitation, with an optional supplementary
non-chronological stratified comparison for those two classes only.


In [ ]:
train_df, val_df, test_df = pipeline.stage1_2_load_clean_split(DATA_DIR)
print("train/val/test flow counts:", len(train_df), len(val_df), len(test_df))
train_df[[config.LABEL_COLUMN, "day"]].value_counts().head(20)


## Stage 4-5: Bidirectional Session Reconstruction & Behavioral Event Encoding

Each partition's flows are grouped into sessions via a **symmetric 5-tuple
key** `{min/max(IP), protocol, min/max(port)}` with a `tau=60s` idle
timeout, then every flow is mapped to one of 10 behavioral tokens via the
explicit, reproducible rule table in `events.RULE_TABLE` (Review §13: rules
must be derivable from flow attributes alone, never the ground-truth label).

Session termination follows the SSH brute-force worked example exactly
(`terminate_on_fin_rst=False` by default) -- see the docstring of
`sessions.reconstruct_sessions` for why the review's own abstract
"FIN/RST ends a session" rule would otherwise shatter that exact worked
example into 8 separate one-flow sessions.


In [ ]:
train_df = pipeline.stage4_5_sessions_and_events(train_df)
val_df = pipeline.stage4_5_sessions_and_events(val_df)
test_df = pipeline.stage4_5_sessions_and_events(test_df)

train_summary = sessions.session_summary(train_df)
print(f"{len(train_df)} flows -> {train_df['session_id'].nunique()} training sessions")
train_summary["n_flows"].hist(bins=30)
plt.xlabel("flows per session"); plt.ylabel("count"); plt.title("Session size distribution (train)")
plt.show()

print("\nBehavioral token distribution (train):")
print(train_df["token"].value_counts())


In [ ]:
# Rule table (Review §13: must be published verbatim)
import pandas as pd
pd.DataFrame(events.RULE_TABLE, columns=["token", "rule"])


## Stage 2 (scaling) + Stage 3: Train-Only Preprocessing & Feature Groups

Median-impute -> variance-filter -> Min-Max scale, **fit on the training
split only** (Review §6), then the 78 CICFlowMeter features are partitioned
into four domain-knowledge groups (Review §7: keyword-matched, MI-ranked
within each group on the training split):

| Group | Feature type | Consumer |
|---|---|---|
| A | flow-statistical | Random Forest |
| B | protocol / communication | shared (RF + XGBoost) |
| C | derived temporal | LSTM context |
| D | TCP behavioral flags | XGBoost |


In [ ]:
pre, X_train, X_val, X_test, class_weights, fsets = pipeline.stage2_3_preprocess_and_group(train_df, val_df, test_df)

print("Scaled feature matrix shapes:", X_train.shape, X_val.shape, X_test.shape)
print("RF features:", len(fsets["rf"]), " | XGBoost features:", len(fsets["xgb"]), " | LSTM context features:", len(fsets["lstm_context"]))

print("\nClass weights W_c = N_train / (K * N_c):")
pd.Series(class_weights).sort_values(ascending=False)


In [ ]:
# Branch B (SMOTE-KNN) -- experimental comparison, RF/XGBoost only, N>=50 classes only.
# This is NOT used by default (Branch A / class weighting is primary, Review §8) --
# run this cell only if you want the A5 ablation comparison.
X_rf_smote, y_rf_smote = preprocessing.smote_knn_resample(X_train[fsets["rf"]], train_df[config.LABEL_COLUMN])
print("Branch A (original) size:", len(X_train), " | Branch B (SMOTE-KNN) size:", len(X_rf_smote))


## Stage 6: Multi-Model Parallel Detection

Three classifiers, each on its designated feature group, trained with
**Branch A class weights** (primary methodology -- Review §8):

- **Random Forest** on Group A+B tabular features.
- **XGBoost** on Group B+D tabular features.
- **Bidirectional LSTM** on Stage 5's session **token sequences** (never
  raw flow rows -- Review §9), trained with weighted cross-entropy, and
  **never** touched by SMOTE-KNN (Review §8/§11).

All three output calibrated probability vectors over the 15 classes; no
hard-decision labels are passed downstream (Stage 6 output contract).


In [ ]:
trained = pipeline.stage6_train_models(X_train, train_df, fsets, class_weights, use_smote_branch_b=False)
print("Models trained. Branch:", trained.branch)


## Stage 8: Frequent & Sequential Pattern Mining

**FP-Growth** mines unordered co-occurring token sets; **PrefixSpan** mines
ordered sequences under a `max_gap` temporal constraint. Review §14/§15:
these are complementary, not redundant -- both feed the sequential-pattern
prior `SP_t`. Mined from **training sessions only**, on **original tokens
only** (never SMOTE-KNN synthetic data).


In [ ]:
train_seqs_raw = events.build_session_sequences(train_df)
train_tokens_only = {sid: [t for t, _ in seq] for sid, seq in train_seqs_raw.items()}

fp_patterns = pattern_mining.mine_fp_growth(train_tokens_only)
seq_patterns = pattern_mining.mine_prefixspan(train_seqs_raw)

print("Top FP-Growth itemsets:")
display(fp_patterns.head(10))
print("\nTop PrefixSpan sequential patterns (token order, temporal_support):")
for pat, sup in seq_patterns[:10]:
    print(f"  {pat}  support={sup:.3f}")


## Stage 9-10: Temporal Attack-State Graph & Sequence Consistency (TC_t)

Nodes are the 10 behavioral tokens themselves (data-driven, **not**
predefined MITRE ATT&CK stages -- Review §16, since CIC-IDS2017 does not
contain every kill-chain stage). Edge weights are an online EMA estimate of
transition probability (`rho=0.9`), built from **training sessions only**,
in chronological order.

`lambda` (the temporal decay constant in `TC_t`) is selected on the
**validation split** by maximising ROC-AUC of `TC_t` as an attack/benign
separator (Review §17: must be empirically validated, not assumed).


In [ ]:
graph = attack_graph.AttackStateGraph().build_from_training(train_seqs_raw)
mean_train_gap = risk.compute_mean_interevent_time(train_seqs_raw)

# session-level dataset needs SOME lambda to compute TC_t the first time;
# we use the mid-point candidate, then re-select lambda on validation and
# recompute (pipeline.run_pipeline does exactly this -- shown explicitly here).
lam0 = config.TC_LAMBDA_CANDIDATES[len(config.TC_LAMBDA_CANDIDATES) // 2]
val_sessions, val_seqs = pipeline.build_session_level_dataset(
    val_df, X_val, trained, fsets, fp_patterns, seq_patterns, graph, lam0, mean_train_gap
)
val_is_attack = val_sessions["is_attack"].to_dict()
lam, auc = attack_graph.select_lambda(val_seqs, graph, val_is_attack)
val_sessions["TC_t"] = [attack_graph.compute_tc_t(val_seqs[s], graph, lam) for s in val_sessions.index]

print(f"Selected lambda={lam}, validation ROC-AUC={auc:.4f}")
val_sessions["TC_t"].hist(bins=30); plt.xlabel("TC_t"); plt.title("Validation TC_t distribution"); plt.show()


In [ ]:
train_sessions, _ = pipeline.build_session_level_dataset(
    train_df, X_train, trained, fsets, fp_patterns, seq_patterns, graph, lam, mean_train_gap
)
test_sessions, test_seqs = pipeline.build_session_level_dataset(
    test_df, X_test, trained, fsets, fp_patterns, seq_patterns, graph, lam, mean_train_gap
)
print("session-level datasets built:", len(train_sessions), len(val_sessions), len(test_sessions))


## Stage 7: Adaptive (Validation-Calibrated, Frozen) Evidence Fusion

`R_t = w_A*P_A + w_B*P_B + w_C*P_C + w_S*SP_t + w_T*TC_t`

Weights are grid-searched / calibrated **on the validation split only**,
then **frozen** for every test session (Review §10: a rolling/live-F1
weighting scheme would need test-time labels -- a leakage bug).


In [ ]:
fusion_weights, val_macro_f1 = pipeline.stage7_calibrate_fusion(val_sessions)
print("Frozen fusion weights:", fusion_weights.as_dict())
print("Validation macro-F1 at these weights:", val_macro_f1)

train_sessions = pipeline.add_fused_predictions(train_sessions, fusion_weights)
val_sessions = pipeline.add_fused_predictions(val_sessions, fusion_weights)
test_sessions = pipeline.add_fused_predictions(test_sessions, fusion_weights)


## Stage 11: Adaptive Risk Decision

`Risk_t = alpha*R_t + beta*SP_t + gamma*TC_t + delta*G_w + epsilon*(1/dt_norm)`,
with `{alpha..epsilon}` the coefficients of a **logistic regression
meta-learner fit on the validation split** (Review §18: hand-picked
coefficients are scientifically indefensible; learned ones are standard
stacking and fully reviewer-defensible).

Three-tier alert system: **BENIGN** (Risk_t < 0.35), **SUSPICIOUS**
(0.35-0.75), **ATTACK** (>= 0.75).


In [ ]:
risk_model = pipeline.stage11_train_risk_model(val_sessions)

train_sessions = pipeline.apply_risk_model(train_sessions, risk_model)
val_sessions = pipeline.apply_risk_model(val_sessions, risk_model)
test_sessions = pipeline.apply_risk_model(test_sessions, risk_model)

print("Test tier distribution:")
print(test_sessions["tier"].value_counts())
test_sessions[["label", "predicted_class", "risk", "tier"]].sample(min(10, len(test_sessions)), random_state=0)


## Stage 24: Evaluation Metrics (primary: Macro-F1, per-class F1, FPR)

Accuracy is reported last / with caution -- BENIGN dominance makes it an
uninformative headline number (Review §24). Heartbleed and Infiltration are
reported per-class, never merged away (Review §4).


In [ ]:
report = metrics.per_class_report(test_sessions["label"], test_sessions["predicted_class"])
display(report)

macro_f1 = metrics.macro_f1(test_sessions["label"], test_sessions["predicted_class"])
fpr = metrics.false_positive_rate(test_sessions["label"], test_sessions["predicted_class"])
print(f"Test Macro-F1: {macro_f1:.4f}   Test FPR: {fpr:.4f}")

pt, lo, hi = metrics.bootstrap_ci(
    test_sessions["label"].values, test_sessions["predicted_class"].values, metrics.macro_f1, n_boot=1000
)
print(f"Macro-F1 95% bootstrap CI: {pt:.4f} [{lo:.4f}, {hi:.4f}]")

metrics.confusion(test_sessions["label"], test_sessions["predicted_class"])


## Stage 12: Streaming Evaluation

60s tumbling windows / 10s stride over the test partition. Model weights
stay **frozen** -- this is streaming *evaluation*, never "online learning"
(Review §19).


In [ ]:
summary_for_stream = test_sessions.reset_index().rename(columns={"index": "session_id"})[
    ["session_id", "start_time", "end_time", "n_flows", "label"]
]
risk_lookup = test_sessions["risk"].to_dict()

def score_session(sid):
    return risk_lookup[sid]

stream_result = streaming.simulate_streaming(summary_for_stream, score_session)
print(f"Throughput: {stream_result.throughput_events_per_sec:.1f} events/sec")
print(f"Mean latency: {stream_result.mean_latency_ms_per_event:.4f} ms/event")
print(streaming.early_detection_stats(stream_result))


## Stage 13: Explainability & Evidence Chain

Three explanation modalities combined into one analyst-facing alert
(Review §20): TreeSHAP (RF/XGBoost), Gradient x Input saliency for the LSTM
(a DeepSHAP-style approximation over token positions), and the graph path
with its learned edge weights.


In [ ]:
attack_alerts = test_sessions[test_sessions["tier"] == "ATTACK"]
if len(attack_alerts) == 0:
    print("No ATTACK-tier sessions in the test set for this run/dataset.")
else:
    sid = attack_alerts.index[0]
    row = test_sessions.loc[sid]
    seq = test_seqs[sid]
    seq_idx = events.sequence_to_indices(seq)

    X_row_rf = X_test.loc[test_df[test_df["session_id"] == sid].index[:1], fsets["rf"]]
    X_row_xgb = X_test.loc[test_df[test_df["session_id"] == sid].index[:1], fsets["xgb"]]
    predicted_idx = config.CLASS_TO_IDX[row["predicted_class"]]

    rf_top = explainability.explain_tree_model(trained.rf, X_row_rf, target_class_idx=None)
    xgb_top = explainability.explain_tree_model(trained.xgb, X_row_xgb, target_class_idx=None)
    lstm_attrib = explainability.explain_lstm_sequence(trained.lstm, seq_idx, predicted_idx)
    path = attack_graph.graph_path_evidence(seq, graph)

    report_text = explainability.build_alert_report(
        sid, row["risk"], row["tier"], row["predicted_class"],
        rf_top, xgb_top, lstm_attrib, path, row["SP_t"], row["TC_t"],
    )
    print(report_text)


## Stage 23: IEEE-Readiness Ablation Study

Five ablations proving the architecture's specific contributions
(Review §23). A1 (no session reconstruction) needs a full separate run with
flows pre-flattened via `ablation.flatten_to_flow_level`; A2/A3/A4 reuse the
already-fitted fusion/risk model cheaply; A5 (class weighting vs SMOTE-KNN)
needs a second `pipeline.run_pipeline(..., use_smote_branch_b=True)` call.


In [ ]:
# A2: no pattern mining (SP_t = 0)
test_no_sp = ablation.rerun_fusion_and_risk(ablation.zero_sp_t(test_sessions), fusion_weights, risk_model)
# A3: no temporal graph (TC_t = G_w = 0)
test_no_graph = ablation.rerun_fusion_and_risk(ablation.zero_temporal_graph(test_sessions), fusion_weights, risk_model)
# A4: static (equal) fusion weights instead of calibrated
test_equal_w = ablation.rerun_fusion_and_risk(test_sessions, ablation.equal_fusion_weights(), risk_model)

comparison = pd.concat([
    ablation.compare_ablation("full_system", test_sessions, "A2_no_pattern_mining", test_no_sp),
    ablation.compare_ablation("full_system", test_sessions, "A3_no_temporal_graph", test_no_graph).iloc[[1]],
    ablation.compare_ablation("full_system", test_sessions, "A4_equal_fusion_weights", test_equal_w).iloc[[1]],
], ignore_index=True)
comparison


## Next steps for the thesis / IEEE submission

- Run **A1** (session-level vs flow-level) by calling `pipeline.run_pipeline`
  a second time after flattening flows with `ablation.flatten_to_flow_level`.
- Run **A5** (Branch A vs Branch B) via
  `pipeline.run_pipeline(DATA_DIR, use_smote_branch_b=True)` and compare
  `ablation.compare_ablation` on the two test-session tables, per-class,
  focusing on Heartbleed/Infiltration rows.
- Run the SMOTE-KNN `k` sensitivity sweep (`config.SMOTE_K_NEIGHBORS`) and
  the class-weighting `K` sensitivity sweep (`config.NUM_CLASSES` in the
  `W_c` formula) called for in the methodology's "IEEE Publication
  Readiness" checklist.
- Report `metrics.mcnemar_test` between the full system's predictions and
  your strongest single-model baseline's predictions, on the test sessions.
- Write up the **zero-training-count caveat** for Heartbleed/Infiltration
  (see the Stage 1-2 markdown cell) explicitly in your Threats to Validity
  section -- it is a stronger, more defensible version of Review §4's
  "low sample count" caveat and is exactly the kind of honest limitation
  reviewers reward rather than penalise.
